# 9.9 Speculative decoding

Speculative decoding lets a small model guess several tokens, then a large model checks them in parallel.

This notebook simulates draft and target token distributions with tiny arrays. The key mechanism is the acceptance ratio $p/q$: speed only counts when proposed tokens are accepted without changing the target distribution.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 909
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def build_speculative_ladder():
    rng = np.random.default_rng(SEED)
    ladders = []
    ladders.append({"name": "D1 one prompt", "target": np.array([0.30, 0.70, 0.90]), "draft": np.array([0.60, 0.40, 1.00])})
    ladders.append({"name": "D2 clean draft", "target": np.array([0.90, 0.80, 0.50, 0.70]), "draft": np.array([0.92, 0.86, 0.55, 0.75])})
    ladders.append({"name": "D3 weak draft", "target": np.array([0.72, 0.35, 0.20, 0.62, 0.41]), "draft": np.array([0.40, 0.75, 0.80, 0.55, 0.76])})
    snippets = ["answer budget", "write headline", "summarize policy", "classify lead", "draft reply", "fix format"]
    target = np.array([0.84, 0.77, 0.64, 0.58, 0.71, 0.66])
    draft = np.array([0.88, 0.81, 0.72, 0.45, 0.69, 0.80])
    ladders.append({"name": "D4 text snippets", "target": target, "draft": draft, "snippets": snippets})
    target_d5 = np.clip(rng.beta(5.0, 2.5, size=24), 0.05, 0.98)
    draft_d5 = np.clip(target_d5 + rng.normal(scale=0.22, size=24), 0.05, 0.98)
    draft_d5[::5] = np.clip(draft_d5[::5] + 0.35, 0.05, 0.98)
    ladders.append({"name": "D5 long context rejections", "target": target_d5, "draft": draft_d5})
    return ladders


## The concept, built once (D1)

The acceptance probability is $$a_i=\min\left(1,\frac{p(x_i\mid x_{\lt i})}{q(x_i\mid x_{\lt i})}\right).$$
The lesson checks $0.30/0.60=0.500$, $0.70/0.40\gt1$ so acceptance is $1.000$, expected accepts $0.9+0.8+0.5=2.2$, and all-accepted probability $0.9\times0.8\times0.5=0.360$.

In [ ]:

def spec_decode(target_probs, draft_probs):
    target_probs = np.asarray(target_probs, dtype=float)
    draft_probs = np.asarray(draft_probs, dtype=float)
    acceptance = np.minimum(1.0, target_probs / draft_probs)
    expected_accepted = float(np.sum(acceptance))
    all_accepted = float(np.prod(acceptance))
    target_passes = 1
    return acceptance, expected_accepted, all_accepted, target_passes

accept_demo, expected_demo, all_demo, passes_demo = spec_decode(np.array([0.30, 0.70]), np.array([0.60, 0.40]))
accept_lesson, expected_lesson, all_lesson, passes_lesson = spec_decode(np.array([0.9, 0.8, 0.5]), np.array([1.0, 1.0, 1.0]))
assert round(float(accept_demo[0]), 3) == 0.500
assert round(float(accept_demo[1]), 3) == 1.000
assert round(expected_lesson, 1) == 2.2
assert round(all_lesson, 3) == 0.360
print(accept_demo, expected_lesson, all_lesson)


The same verifier can score a whole draft block. The metric is accepted tokens per target pass, not tokens proposed by the draft.

In [ ]:

target_block = np.array([0.9, 0.8, 0.5, 0.7])
draft_block = np.array([1.0, 1.0, 1.0, 0.7])
acceptance_block, accepted_block, all_block, passes_block = spec_decode(target_block, draft_block)
assert accepted_block <= len(target_block)
print("acceptance", acceptance_block)
print("accepted per target pass", accepted_block / passes_block)


## The dataset ladder

D1 is the worked prompt; later rungs add clean drafts, weak drafts, text snippets, and a longer D5 context with many rejection opportunities.

In [ ]:

ladder = build_speculative_ladder()
for rung in ladder:
    print(rung["name"], "tokens", len(rung["target"]), "target mean", np.mean(rung["target"]), "draft mean", np.mean(rung["draft"]))
    print("sample target", rung["target"][:4])
    print("sample draft", rung["draft"][:4])


## Run the same speculative decoder across D1-D5

In [ ]:

results = []
for rung in ladder:
    acceptance, expected_accepted, all_accepted, target_passes = spec_decode(rung["target"], rung["draft"])
    metric = expected_accepted / target_passes
    results.append({"name": rung["name"], "acceptance": acceptance, "metric": metric, "all": all_accepted, "proposed": len(acceptance)})

print("rung | accepted_per_target_pass | proposed | all_accepted_probability")
for result in results:
    print(f"{result['name']} | {result['metric']:.3f} | {result['proposed']} | {result['all']:.3f}")


## Results visualization

Small multiples show token-level acceptance. The summary curve shows accepted tokens per target pass.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for ax, result in zip(axes, results):
    ax.bar(np.arange(len(result["acceptance"])), result["acceptance"])
    ax.set_ylim(0.0, 1.05)
    ax.set_title(result["name"].split()[0])
    ax.set_xlabel("draft token")
    ax.set_ylabel("accept prob")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(results) + 1), [item["metric"] for item in results], marker="o")
ax.set_xticks(range(1, len(results) + 1))
ax.set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
ax.set_ylabel("accepted tokens / target pass")
ax.set_title("Speed depends on accepted tokens")
plt.show()


## Pitfall on the hardest rung

Pitfall: counting proposed tokens rather than accepted tokens overstates speedup. D5 makes that gap visible.

In [ ]:

d5 = ladder[-1]
acceptance, expected_accepted, all_accepted, target_passes = spec_decode(d5["target"], d5["draft"])
wrong_speed = len(acceptance) / target_passes
correct_speed = expected_accepted / target_passes
first_rejection = int(np.argmin(acceptance))
print("wrong proposed-token speed", wrong_speed)
print("correct accepted-token speed", correct_speed)
print("lowest acceptance token", first_rejection, acceptance[first_rejection])


## Evaluate it + Practice

- Metric: accepted tokens per target pass.
- No-skill baseline: ordinary decoding accepts exactly one target token per large-model pass.
- Cheap sanity check: the worked ratios must give 0.500 and 1.000 acceptance.
- Ablation: replace $p/q$ with greedy acceptance and watch the target-distribution guarantee disappear.
- Failure signals: high proposed-token counts, low acceptance, or all-accepted probability near zero.

Practice prompts:


**Practice.** Make D5 draft probabilities closer to target probabilities and re-plot speed.

**Practice.** Lower the draft quality on every fifth token and explain the all-accepted curve.

**Practice.** Compare accepted tokens per pass with wall-clock cost if the draft model costs 20% of target.